In [4]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
credits = pd.read_csv("tmdb_5000_credits.csv")
movies = pd.read_csv("tmdb_5000_movies.csv")
df = movies.merge(credits, on="title")
features = ['genres', 'keywords', 'cast', 'crew', 'overview']
df = df[features + ['title']]
for feature in features:
    df[feature] = df[feature].fillna('[]')
def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]
df['genres'] = df['genres'].apply(convert)
df['keywords'] = df['keywords'].apply(convert)
def get_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L
df['cast'] = df['cast'].apply(get_cast)
def get_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

df['crew'] = df['crew'].apply(get_director)
def clean(text):
    return [i.replace(" ", "") for i in text]

for feature in ['genres', 'keywords', 'cast', 'crew']:
    df[feature] = df[feature].apply(clean)
df['combined'] = df['genres'] + df['keywords'] + df['cast'] + df['crew']
df['combined'] = df['combined'].apply(lambda x: " ".join(x))
tfidf = TfidfVectorizer(stop_words='english')
vector = tfidf.fit_transform(df['combined'])
similarity = cosine_similarity(vector)
def recommend(movie):
    index = df[df['title'] == movie].index[0]
    distances = list(enumerate(similarity[index]))
    movies_list = sorted(distances, key=lambda x: x[1], reverse=True)[1:8]
    
    for i in movies_list:
        print(df.iloc[i[0]].title)
recommend("John Carter")


Mission to Mars
My Favorite Martian
Miss Julie
Spaced Invaders
The Last Days on Mars
The Adventurer: The Curse of the Midas Box
Mars Needs Moms


In [5]:
import pickle

# Save the similarity matrix and dataframe
pickle.dump(similarity, open("similarity.pkl", "wb"))
pickle.dump(df, open("movies.pkl", "wb"))